# p53 Mutant Structure Prediction & MD Simulation

This notebook uses the **ESMFold API** (no local installation needed) to predict mutant structures,
then runs MD simulation using OpenMM.

## Key Points
- ESMFold API requires NO dependencies - just HTTP requests
- Structures are properly predicted (not just side-chain swaps like EvoEF2 BuildMutant)
- Suitable for validating rescue mutations

In [ ]:
# @title 1. Install Dependencies (Run Once)
!pip install -q openmm pdbfixer mdtraj py3Dmol requests biopython

In [ ]:
# @title 2. Imports
import requests
import time
import os
from pathlib import Path

# OpenMM
from openmm import *
from openmm.app import *
from openmm.unit import *

# Analysis
import mdtraj as md
import numpy as np
import matplotlib.pyplot as plt

# Visualization
import py3Dmol

print("All imports successful!")

In [ ]:
# @title 3. ESMFold API Functions

# Primary: ESMFold API (Meta's ESM Atlas)
ESMFOLD_API_URL = "https://api.esmatlas.com/foldSequence/v1/pdb/"

# Fallback: Hugging Face Inference API
HF_API_URL = "https://api-inference.huggingface.co/models/facebook/esmfold_v1"

def predict_structure_esmfold(sequence: str, max_retries: int = 3, use_hf_fallback: bool = True) -> str:
    """
    Predict protein structure using ESMFold API.
    
    Args:
        sequence: Amino acid sequence (single letter codes)
        max_retries: Number of retry attempts per API
        use_hf_fallback: Whether to try Hugging Face API as fallback
        
    Returns:
        PDB string
    """
    # Try primary API (ESM Atlas)
    print("Trying ESMFold API (ESM Atlas)...")
    for attempt in range(max_retries):
        try:
            print(f"  Attempt {attempt + 1}/{max_retries}...")
            response = requests.post(
                ESMFOLD_API_URL,
                data=sequence,
                headers={'Content-Type': 'text/plain'},
                timeout=300  # 5 minute timeout
            )
            
            if response.status_code == 200:
                print("  Success!")
                return response.text
            elif response.status_code == 503:
                print(f"  Server busy, waiting 30s...")
                time.sleep(30)
            elif response.status_code == 404:
                print(f"  API endpoint not found (404) - may be deprecated")
                break  # Don't retry, try fallback instead
            else:
                print(f"  Error: HTTP {response.status_code}")
                if response.text:
                    print(f"  Response: {response.text[:200]}")
                time.sleep(10)
                
        except requests.Timeout:
            print(f"  Timeout, retrying...")
            time.sleep(10)
        except requests.ConnectionError as e:
            print(f"  Connection error: {e}")
            time.sleep(10)
    
    # Try Hugging Face fallback
    if use_hf_fallback:
        print("\nTrying Hugging Face Inference API as fallback...")
        for attempt in range(max_retries):
            try:
                print(f"  Attempt {attempt + 1}/{max_retries}...")
                response = requests.post(
                    HF_API_URL,
                    json={"inputs": sequence},
                    headers={'Content-Type': 'application/json'},
                    timeout=600  # 10 minute timeout (HF can be slow)
                )
                
                if response.status_code == 200:
                    # HF returns JSON with PDB in a specific format
                    result = response.json()
                    if isinstance(result, str):
                        print("  Success!")
                        return result
                    elif isinstance(result, dict) and 'generated_text' in result:
                        print("  Success!")
                        return result['generated_text']
                    elif isinstance(result, list) and len(result) > 0:
                        print("  Success!")
                        return result[0] if isinstance(result[0], str) else str(result[0])
                    else:
                        print(f"  Unexpected response format: {type(result)}")
                        time.sleep(10)
                elif response.status_code == 503:
                    # Model is loading
                    wait_time = response.json().get('estimated_time', 60)
                    print(f"  Model loading, waiting {wait_time}s...")
                    time.sleep(min(wait_time, 120))
                else:
                    print(f"  Error: HTTP {response.status_code}")
                    time.sleep(10)
                    
            except requests.Timeout:
                print(f"  Timeout, retrying...")
                time.sleep(10)
            except Exception as e:
                print(f"  Error: {e}")
                time.sleep(10)
    
    raise RuntimeError(
        f"ESMFold prediction failed after trying all APIs.\n"
        f"Alternatives:\n"
        f"  1. Try again later (servers may be overloaded)\n"
        f"  2. Use ColabFold: https://colab.research.google.com/github/sokrypton/ColabFold\n"
        f"  3. Use local ESMFold with 'pip install fair-esm' (requires GPU)"
    )


def apply_mutation(sequence: str, mutation: str) -> str:
    """
    Apply a mutation to a sequence.
    
    Args:
        sequence: Original amino acid sequence
        mutation: Mutation string like 'R175H' (1-indexed)
        
    Returns:
        Mutated sequence
    """
    wt_aa = mutation[0]
    pos = int(mutation[1:-1]) - 1  # Convert to 0-indexed
    mut_aa = mutation[-1]
    
    if sequence[pos] != wt_aa:
        raise ValueError(f"Expected {wt_aa} at position {pos+1}, found {sequence[pos]}")
    
    return sequence[:pos] + mut_aa + sequence[pos+1:]


print("ESMFold API functions defined (with Hugging Face fallback).")

In [ ]:
# @title 4. p53 Sequence (Core Domain, residues 94-312)

# Full p53 sequence for reference
P53_FULL = (
    "MEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLSPDDIEQWFTEDPGP"
    "DEAPRMPEAAPPVAPAPAAPTPAAPAPAPSWPLSSSVPSQKTYQGSYGFRLGFLHSGTAK"
    "SVTCTYSPALNKMFCQLAKTCPVQLWVDSTPPPGTRVRAMAIYKQSQHMTEVVRRCPHHE"
    "RCSDSDGLAPPQHLIRVEGNLRVEYLDDRNTFRHSVVVPYEPPEVGSDCTTIHYNYMCNS"
    "SCMGGMNRRPILTIITLEDSSGNLLGRNSFEVRVCACPGRDRRTEEENLRKKGEPHHELP"
    "PGSTKRALPNNTSSSPQPKKKPLDGEYFTLQIRGRERFEMFRELNEALELKDAQAGKEPG"
    "GSRAHSSHLKSKKGQSTSRHKKLMFKTEGPDSD"
)

# Core domain (most commonly used for structural studies)
# Residues 94-312 (0-indexed: 93:312)
P53_CORE = P53_FULL[93:312]  # 219 residues

print(f"p53 core domain length: {len(P53_CORE)} residues")
print(f"Sequence: {P53_CORE[:50]}...")

In [ ]:
# @title 5. Configuration

# === EDIT THESE ===
TARGET_MUTATION = "R175H"  # Destabilizing mutation (cancer hotspot)
RESCUE_MUTATIONS = ["N239Y"]  # Putative rescue mutations

# MD settings
EQUILIBRATION_STEPS = 50000  # 100 ps NPT equilibration
PRODUCTION_STEPS = 500000   # 1 ns production (increase for better sampling)
TIMESTEP = 2.0  # femtoseconds

# === END EDIT ===

# Build combined mutation list
all_mutations = [TARGET_MUTATION] + RESCUE_MUTATIONS
print(f"Target: {TARGET_MUTATION}")
print(f"Rescue: {RESCUE_MUTATIONS}")
print(f"Combined: {'+'.join(all_mutations)}")

In [ ]:
# @title 6. Predict Mutant Structure with ESMFold

# Create output directory
os.makedirs("structures", exist_ok=True)

# Apply all mutations to core domain
# Note: mutation positions are relative to FULL sequence (1-indexed)
# We need to adjust for core domain (starts at residue 94)

mutant_seq = P53_CORE
core_start = 94  # First residue number in core domain

for mut in all_mutations:
    wt_aa = mut[0]
    pos = int(mut[1:-1])  # Full sequence position (1-indexed)
    mut_aa = mut[-1]
    
    # Convert to core domain position (0-indexed)
    core_pos = pos - core_start
    
    if core_pos < 0 or core_pos >= len(mutant_seq):
        raise ValueError(f"Position {pos} is outside core domain (94-312)")
    
    if mutant_seq[core_pos] != wt_aa:
        raise ValueError(f"Expected {wt_aa} at position {pos}, found {mutant_seq[core_pos]}")
    
    mutant_seq = mutant_seq[:core_pos] + mut_aa + mutant_seq[core_pos+1:]
    print(f"Applied {mut}: pos {core_pos} in core domain")

print(f"\nMutant sequence: {mutant_seq[:50]}...")

# Predict structure
print("\nPredicting structure with ESMFold...")
pdb_string = predict_structure_esmfold(mutant_seq)

# Save
mutant_name = "_".join(all_mutations)
pdb_path = f"structures/{mutant_name}_esmfold.pdb"
with open(pdb_path, 'w') as f:
    f.write(pdb_string)
    
print(f"\nSaved to {pdb_path}")

In [ ]:
# @title 7. Visualize Predicted Structure

with open(pdb_path, 'r') as f:
    pdb_string = f.read()

view = py3Dmol.view(width=800, height=600)
view.addModel(pdb_string, 'pdb')
view.setStyle({'cartoon': {'color': 'spectrum'}})

# Highlight mutation sites
for mut in all_mutations:
    pos = int(mut[1:-1]) - core_start + 1  # PDB is 1-indexed
    view.addStyle({'resi': pos}, {'stick': {'color': 'red'}})

view.zoomTo()
view.show()

In [ ]:
# @title 8. Prepare System for MD

from pdbfixer import PDBFixer

# Fix structure
print("Fixing structure with PDBFixer...")
fixer = PDBFixer(filename=pdb_path)
fixer.findMissingResidues()
fixer.findMissingAtoms()
fixer.addMissingAtoms()
fixer.addMissingHydrogens(7.0)

# Save fixed
fixed_path = f"structures/{mutant_name}_fixed.pdb"
with open(fixed_path, 'w') as f:
    PDBFile.writeFile(fixer.topology, fixer.positions, f)
print(f"Saved fixed structure to {fixed_path}")

# Create system
print("\nCreating simulation system...")
pdb = PDBFile(fixed_path)
forcefield = ForceField('amber14-all.xml', 'amber14/tip3pfb.xml')

# Add solvent
modeller = Modeller(pdb.topology, pdb.positions)
modeller.addSolvent(forcefield, model='tip3p', padding=1.0*nanometer, ionicStrength=0.15*molar)

# Save solvated
solvated_path = f"structures/{mutant_name}_solvated.pdb"
with open(solvated_path, 'w') as f:
    PDBFile.writeFile(modeller.topology, modeller.positions, f)
print(f"Saved solvated system to {solvated_path}")

# Create system
system = forcefield.createSystem(
    modeller.topology,
    nonbondedMethod=PME,
    nonbondedCutoff=1.0*nanometer,
    constraints=HBonds
)

print(f"System has {system.getNumParticles()} particles")

In [ ]:
# @title 9. Energy Minimization

print("Setting up integrator and simulation...")
integrator = LangevinMiddleIntegrator(300*kelvin, 1/picosecond, TIMESTEP*femtoseconds)
simulation = Simulation(modeller.topology, system, integrator)
simulation.context.setPositions(modeller.positions)

# Minimize
print("Running energy minimization...")
initial_energy = simulation.context.getState(getEnergy=True).getPotentialEnergy()
print(f"  Initial energy: {initial_energy}")

simulation.minimizeEnergy(maxIterations=1000)

final_energy = simulation.context.getState(getEnergy=True).getPotentialEnergy()
print(f"  Final energy: {final_energy}")

# Save minimized
min_path = f"structures/{mutant_name}_minimized.pdb"
positions = simulation.context.getState(getPositions=True).getPositions()
with open(min_path, 'w') as f:
    PDBFile.writeFile(simulation.topology, positions, f)
print(f"Saved minimized structure to {min_path}")

In [ ]:
# @title 10. NPT Equilibration

# Add barostat for NPT
system.addForce(MonteCarloBarostat(1*bar, 300*kelvin))

# Reinitialize with barostat
integrator = LangevinMiddleIntegrator(300*kelvin, 1/picosecond, TIMESTEP*femtoseconds)
simulation = Simulation(modeller.topology, system, integrator)
simulation.context.setPositions(positions)
simulation.context.setVelocitiesToTemperature(300*kelvin)

# Equilibrate
print(f"Running NPT equilibration ({EQUILIBRATION_STEPS} steps, {EQUILIBRATION_STEPS * TIMESTEP / 1000:.1f} ps)...")
simulation.step(EQUILIBRATION_STEPS)

# Save equilibrated
eq_path = f"structures/{mutant_name}_equilibrated.pdb"
positions = simulation.context.getState(getPositions=True).getPositions()
with open(eq_path, 'w') as f:
    PDBFile.writeFile(simulation.topology, positions, f)
print(f"Saved equilibrated structure to {eq_path}")

In [ ]:
# @title 11. Production MD

os.makedirs("trajectories", exist_ok=True)

# Add reporters
traj_path = f"trajectories/{mutant_name}_traj.dcd"
simulation.reporters.append(DCDReporter(traj_path, 5000))  # Save every 10 ps
simulation.reporters.append(StateDataReporter(
    f"trajectories/{mutant_name}_log.csv", 5000,
    step=True, time=True, potentialEnergy=True, temperature=True,
    progress=True, remainingTime=True, speed=True,
    totalSteps=PRODUCTION_STEPS
))

# Run production
print(f"Running production MD ({PRODUCTION_STEPS} steps, {PRODUCTION_STEPS * TIMESTEP / 1e6:.1f} ns)...")
print("This will take a while...")

simulation.step(PRODUCTION_STEPS)

print(f"\nDone! Trajectory saved to {traj_path}")

In [ ]:
# @title 12. Analyze RMSD

# Load trajectory
traj = md.load(traj_path, top=eq_path)
print(f"Loaded trajectory: {traj.n_frames} frames, {traj.n_atoms} atoms")

# Select protein atoms only (not water)
protein = traj.atom_slice(traj.topology.select('protein'))
print(f"Protein atoms: {protein.n_atoms}")

# Calculate RMSD vs first frame
rmsd = md.rmsd(protein, protein, 0) * 10  # Convert nm to Angstrom

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
time_ns = np.arange(len(rmsd)) * TIMESTEP * 5000 / 1e6  # Convert to ns
ax.plot(time_ns, rmsd, 'b-', linewidth=0.5)
ax.axhline(y=2.0, color='g', linestyle='--', label='Stable threshold (2 Å)')
ax.axhline(y=3.0, color='orange', linestyle='--', label='Moderate threshold (3 Å)')
ax.axhline(y=5.0, color='r', linestyle='--', label='Unstable threshold (5 Å)')

ax.set_xlabel('Time (ns)')
ax.set_ylabel('RMSD (Å)')
ax.set_title(f'{mutant_name} - Backbone RMSD vs Time')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"trajectories/{mutant_name}_rmsd.png", dpi=150)
plt.show()

# Statistics
print(f"\n=== RMSD Statistics ===")
print(f"Mean RMSD: {np.mean(rmsd):.2f} Å")
print(f"Final RMSD: {rmsd[-1]:.2f} Å")
print(f"Max RMSD: {np.max(rmsd):.2f} Å")
print(f"Std RMSD: {np.std(rmsd):.2f} Å")

# Verdict
final_rmsd = rmsd[-1]
if final_rmsd < 2.5:
    print(f"\n✅ STABLE: Final RMSD = {final_rmsd:.2f} Å")
elif final_rmsd < 4.0:
    print(f"\n⚠️ MODERATE: Final RMSD = {final_rmsd:.2f} Å")
else:
    print(f"\n❌ UNSTABLE: Final RMSD = {final_rmsd:.2f} Å (structure unfolding)")

In [ ]:
# @title 13. Analyze RMSF (Flexibility)

# Calculate RMSF per residue
# First, superpose all frames to the first frame
protein_aligned = protein.superpose(protein, 0)

# Calculate RMSF for CA atoms (around average structure)
ca_indices = protein_aligned.topology.select('name CA')
rmsf = md.rmsf(protein_aligned, atom_indices=ca_indices) * 10  # nm to Å

# Get residue numbers
residue_numbers = [protein_aligned.topology.atom(i).residue.resSeq for i in ca_indices]

# Plot
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(residue_numbers, rmsf, 'b-', linewidth=1)
ax.fill_between(residue_numbers, 0, rmsf, alpha=0.3)

# Mark mutation sites
for mut in all_mutations:
    pos = int(mut[1:-1])
    ax.axvline(x=pos, color='red', linestyle='--', alpha=0.7, label=f'{mut}')

ax.set_xlabel('Residue Number')
ax.set_ylabel('RMSF (Å)')
ax.set_title(f'{mutant_name} - Per-Residue Flexibility')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"trajectories/{mutant_name}_rmsf.png", dpi=150)
plt.show()

# Find most flexible regions
print("\n=== Most Flexible Regions ===")
sorted_indices = np.argsort(rmsf)[::-1]
for i in range(min(5, len(sorted_indices))):
    idx = sorted_indices[i]
    print(f"  Residue {residue_numbers[idx]}: RMSF = {rmsf[idx]:.2f} Å")

In [ ]:
# @title 14. Summary & Conclusions

print("="*60)
print(f"SIMULATION SUMMARY: {mutant_name}")
print("="*60)
print(f"\nMutations:")
print(f"  Target (destabilizing): {TARGET_MUTATION}")
print(f"  Rescue mutations: {RESCUE_MUTATIONS}")

print(f"\nSimulation:")
print(f"  Equilibration: {EQUILIBRATION_STEPS * TIMESTEP / 1e6:.3f} ns")
print(f"  Production: {PRODUCTION_STEPS * TIMESTEP / 1e6:.3f} ns")
print(f"  Total frames: {traj.n_frames}")

print(f"\nStability Assessment:")
print(f"  Mean RMSD: {np.mean(rmsd):.2f} Å")
print(f"  Final RMSD: {rmsd[-1]:.2f} Å")
print(f"  Max RMSD: {np.max(rmsd):.2f} Å")

# Interpretation
print(f"\nInterpretation:")
if rmsd[-1] < 2.5:
    print("  ✅ The rescue mutation(s) appear to STABILIZE the protein.")
    print("  The structure maintains its fold throughout the simulation.")
elif rmsd[-1] < 4.0:
    print("  ⚠️ Moderate stability - some conformational changes observed.")
    print("  Consider longer simulations or additional mutations.")
else:
    print("  ❌ The protein shows significant structural drift.")
    print("  The rescue mutation(s) may not be sufficient.")
    print("  Consider different rescue mutations.")

print(f"\nFiles saved:")
print(f"  - structures/{mutant_name}_esmfold.pdb (ESMFold prediction)")
print(f"  - structures/{mutant_name}_equilibrated.pdb (After equilibration)")
print(f"  - trajectories/{mutant_name}_traj.dcd (MD trajectory)")
print(f"  - trajectories/{mutant_name}_rmsd.png (RMSD plot)")
print(f"  - trajectories/{mutant_name}_rmsf.png (RMSF plot)")

## Next Steps

1. **Compare with WT**: Run the same simulation on wild-type p53 to establish baseline
2. **Compare with mutant-only**: Run on R175H without rescue to see destabilization
3. **Longer simulations**: Extend to 10-50 ns for more reliable statistics
4. **Try different rescues**: Test multiple rescue mutations from the Pareto front
5. **Download files**: Save trajectory for detailed analysis in PyMOL/VMD